In [0]:
# MAGIC ## 1. Setup
from pyspark.sql import functions as F

SOURCE_PATH = "/databricks-datasets/retail-org/loyalty_segments/"
TARGET_TABLE = "retail_dev.bronze.bronze_loyalty_segments"

In [0]:
# MAGIC ## 2. Schema e Volume
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(SOURCE_PATH)
)

print("=== SCHEMA ===")
df.printSchema()

print(f"\n=== VOLUME ===")
print(f"Total de linhas:   {df.count()}")
print(f"Total de colunas:  {len(df.columns)}")
print(f"Colunas:           {df.columns}")


In [0]:
# MAGIC ## 3. Amostra
# MAGIC > Tabela pequena — exibindo todas as linhas.

# COMMAND ----------

display(df)


In [0]:
# MAGIC ## 4. Qualidade — Nulos por Coluna

# COMMAND ----------

display(
    df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])
)

In [0]:
# MAGIC ## 5. Análise
# MAGIC > `loyalty_segments` é uma tabela de **definição de faixas** — define os thresholds de cada segmento.
# MAGIC > Não contém `customer_id`. O vínculo com clientes precisa ser investigado na tabela `customers`.

# COMMAND ----------

# MAGIC %md
# MAGIC ### Segmentos disponíveis e seus thresholds

# COMMAND ----------

display(
    df.groupBy("loyalty_segment_id", "loyalty_segment_description")
    .agg(
        F.min("unit_threshold").alias("threshold_minimo"),
        F.max("unit_threshold").alias("threshold_maximo"),
        F.count("*").alias("total_linhas"),
    )
    .orderBy("loyalty_segment_id")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Range de validade dos segmentos

# COMMAND ----------

display(
    df.select(
        F.min("valid_from").alias("valid_from_minimo"),
        F.max("valid_from").alias("valid_from_maximo"),
        F.min("valid_to").alias("valid_to_minimo"),
        F.max("valid_to").alias("valid_to_maximo"),
    )
)

# COMMAND ----------


In [0]:
# MAGIC ## 6. Ingestão → Delta
# MAGIC Escrita na camada bronze — estrutura raw preservada, zero transformações.

# COMMAND ----------

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Tabela escrita: {TARGET_TABLE}")

# COMMAND ----------


In [0]:
# MAGIC ## 7. Verificação

# COMMAND ----------

df_check = spark.table(TARGET_TABLE)

print(f"Linhas na tabela: {df_check.count()}")
print(f"Colunas:          {df_check.columns}")

# COMMAND ----------

display(df_check)
